# RAG-пайплайн с Rejection Sampling и SFT


RAG-пайплайн: retrieval, генерация ответов, оценка кандидатов reward-моделью, rejection sampling и короткий SFT-цикл генератора.


Используется базовый набор библиотек для работы с датасетом, retrieval, генерацией, reward-моделью и коротким SFT через `trl.SFTTrainer`.


In [ ]:
!pip -q install -U datasets sentence-transformers transformers accelerate pandas nltk trl

In [2]:
import gc
import json
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, load_dataset
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
)
from transformers.trainer_utils import get_last_checkpoint
from trl import SFTConfig, SFTTrainer



In [3]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def has_mps_backend() -> bool:
    return (
        hasattr(torch, "backends")
        and hasattr(torch.backends, "mps")
        and torch.backends.mps.is_available()
        and hasattr(torch, "mps")
    )


def has_cuda_bf16_support() -> bool:
    return (
        torch.cuda.is_available()
        and hasattr(torch.cuda, "is_bf16_supported")
        and torch.cuda.is_bf16_supported()
    )


def select_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if has_mps_backend():
        return "mps"
    return "cpu"


def cleanup_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if has_mps_backend() and hasattr(torch.mps, "empty_cache"):
        torch.mps.empty_cache()


def use_bf16_training(device: str) -> bool:
    return device == "cuda" and has_cuda_bf16_support()


def use_fp16_training(device: str) -> bool:
    return device == "cuda" and not use_bf16_training(device)


def use_gradient_checkpointing(device: str) -> bool:
    return device in {"cuda", "mps"}


def use_pin_memory(device: str) -> bool:
    return device == "cuda"


def get_mixed_precision_mode(device: str) -> str:
    if use_bf16_training(device):
        return "bf16"
    if use_fp16_training(device):
        return "fp16"
    return "off"


def pick_torch_dtype(device: str):
    if device == "cuda":
        if has_cuda_bf16_support():
            return torch.bfloat16
        return torch.float16
    return torch.float32


ARTIFACTS_DIR = Path("./rag_artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

GEN_DEVICE = select_device()
EMBED_DEVICE = "cpu"
REWARD_DEVICE = "cpu"

print(f"generator device: {GEN_DEVICE}")
print(f"mixed precision: {get_mixed_precision_mode(GEN_DEVICE)}")
print(f"retrieval device: {EMBED_DEVICE}")
print(f"reward device: {REWARD_DEVICE}")
print(f"artifacts dir: {ARTIFACTS_DIR}")


generator device: cuda
mixed precision: bf16
retrieval device: cpu
reward device: cpu
artifacts dir: rag_artifacts



`CONFIG` хранит названия моделей и основные параметры запуска, чтобы их было удобно менять в одном месте.


In [4]:
CONFIG = {
    "subset_size": 6, # кол-во документов
    "n_candidates": 5,
    "retrieval_model_name": "intfloat/e5-small-v2",
    "generation_model_name": "Qwen/Qwen2.5-0.5B-Instruct",
    "reward_model_name": "Skywork/Skywork-Reward-V2-Qwen3-0.6B",
    "seed": 42,
}

set_seed(CONFIG["seed"])
print(CONFIG)


{'subset_size': 6, 'n_candidates': 5, 'retrieval_model_name': 'intfloat/e5-small-v2', 'generation_model_name': 'Qwen/Qwen2.5-0.5B-Instruct', 'reward_model_name': 'Skywork/Skywork-Reward-V2-Qwen3-0.6B', 'seed': 42}



Для эксперимента используется датасет `neural-bridge/rag-dataset-12000` с вопросами, ответами и длинными контекстами. Контексты разбиваются на чанки с перекрытием, чтобы retrieval мог искать релевантные фрагменты, а не работать с длинным сырым текстом целиком.


In [5]:
wikiq = load_dataset("neural-bridge/rag-dataset-12000")


def pick_long_context_examples(split, subset_size: int) -> pd.DataFrame:
    frame = split.to_pandas()
    frame["context_words"] = frame["context"].str.split().str.len()
    frame = (frame
             .sort_values("context_words", ascending=False)
             .head(subset_size)
             .reset_index(drop=True)
             )
    frame["example_id"] = np.arange(len(frame))
    return frame


train_df = pick_long_context_examples(wikiq["train"], CONFIG["subset_size"])
train_df[["example_id", "question", "answer", "context_words"]].head(3)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-9df3a936e1f631(…):   0%|          | 0.00/23.1M [00:00<?, ?B/s]

data/test-00000-of-00001-af2a9f454ad1b8a(…):   0%|          | 0.00/5.79M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9600 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2400 [00:00<?, ? examples/s]

,example_id,question,answer,context_words
0,0,What are some of the primary areas where machi...,The primary areas where machine learning can p...,1283
1,1,What are some methods used to align teeth?,Some methods used to align teeth include the u...,1280
2,2,What is the rating of the Cheap School Movie R...,The Cheap School Movie Review is rated 4/5 bas...,1270


`Chunker` разбивает документы на фрагменты фиксированного размера с заданным overlap. Перекрытие помогает не терять факты, которые могут оказаться на границе двух соседних чанков.


In [6]:
class Chunker:
    def __init__(self, chunk_words: int = 100, overlap: int = 20):
        if chunk_words <= 0:
            raise ValueError("chunk_words must be positive")
        if overlap < 0 or overlap >= chunk_words:
            raise ValueError("overlap must be between 0 and chunk_words - 1")
        self.chunk_words = chunk_words
        self.overlap = overlap
        self.step = chunk_words - overlap

    def split_text_to_chunks(self, text: str) -> list[str]:
        words = text.split()
        chunks = []
        for start in range(0, len(words), self.step):
            chunk_words = words[start:start + self.chunk_words]
            if chunk_words:
                chunks.append(" ".join(chunk_words))
        return chunks

    def get_chunked_list(self, texts: list[str]) -> list[str]:
        chunked_docs = []
        for text in texts:
            chunked_docs.extend(self.split_text_to_chunks(text))
        return chunked_docs


In [7]:
chunker = Chunker(
    chunk_words=100,
    overlap=20,
)

train_full_docs = train_df["context"].tolist()
train_queries = train_df["question"].tolist()
train_answers = train_df["answer"].tolist()

train_chunked_docs = chunker.get_chunked_list(train_full_docs)
print(f"documents: {len(train_full_docs)}")
print(f"chunks: {len(train_chunked_docs)}")
print(train_chunked_docs[0][:400])


documents: 6
chunks: 97
Machine Learning is an application that comes under the hood of Artificial Intelligence. The primary aim of machine learning is to enable software applications to become exact and precise in forecasting the results. Under Machine Learning, a computer tends to learn automatically without any sort of human intercession. According to experts, who are witnessing the current trends, Artificial Intellig


Для retrieval используется `intfloat/e5-small-v2` — компактный bi-encoder для кодирования запросов и документов. Для моделей семейства E5 важно добавлять префиксы `query:` и `passage:`, чтобы embeddings строились в ожидаемом формате.


In [8]:
model = SentenceTransformer(CONFIG['retrieval_model_name'], device=EMBED_DEVICE)

query_embeddings = model.encode([
    "query: What is TSNE?",
    "query: Who is Laurens van der Maaten?",
])
doc_embeddings = model.encode([
    "passage: TSNE is a dimensionality reduction algorithm created by Laurens van Der Maaten",
])
print(query_embeddings.shape, doc_embeddings.shape)
# (2, 384) (1, 384)

similarities = model.similarity(query_embeddings, doc_embeddings)
print(similarities)
# tensor([[0.8971],
#         [0.8112]])
del model

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

(2, 384) (1, 384)
tensor([[0.8971],
        [0.8112]])


In [9]:
class Encoder:
    def __init__(self, model_name: str = CONFIG["retrieval_model_name"], device: str = EMBED_DEVICE):
        self.model = SentenceTransformer(model_name, device=device)

    def encode_query(self, texts: list[str]) -> np.ndarray:
        prefixed = [f"query: {text.strip()}" for text in texts]
        embeddings = self.model.encode(
            prefixed,
            batch_size=32,
            show_progress_bar=False,
            normalize_embeddings=True,
        )
        return np.asarray(embeddings)

    def encode_docs(self, texts: list[str]) -> np.ndarray:
        prefixed = [f"passage: {text.strip()}" for text in texts]
        embeddings = self.model.encode(
            prefixed,
            batch_size=32,
            show_progress_bar=False,
            normalize_embeddings=True,
        )
        return np.asarray(embeddings)


class VectorSearchEngine:
    def __init__(self, init_base: list[str], encoder: Encoder):
        self.encoder = encoder
        self.text_index = list(init_base)
        self.doc_embeddings = None
        if self.text_index:
            self.doc_embeddings = self.encoder.encode_docs(self.text_index)

    def insert_doc(self, doc: str) -> None:
        new_embedding = self.encoder.encode_docs([doc])
        self.text_index.append(doc)
        if self.doc_embeddings is None:
            self.doc_embeddings = new_embedding
        else:
            self.doc_embeddings = np.vstack([self.doc_embeddings, new_embedding])

    def get_k_most_similar(self, query: str, k: int = 3) -> tuple[list[str], list[float]]:
        if not self.text_index or self.doc_embeddings is None:
            raise ValueError("The search index is empty")
        query_embedding = self.encoder.encode_query([query])[0]
        scores = self.doc_embeddings @ query_embedding
        k = min(k, len(self.text_index))
        top_indices = np.argsort(scores)[-k:][::-1]
        top_texts = [self.text_index[idx] for idx in top_indices]
        top_scores = [float(scores[idx]) for idx in top_indices]
        return top_texts, top_scores


In [10]:
encoder = Encoder(model_name=CONFIG["retrieval_model_name"])
vse = VectorSearchEngine(train_chunked_docs, encoder)

retrieved_docs, retrieved_scores = vse.get_k_most_similar(train_queries[0], k=2)
print(train_queries[0])
print(retrieved_scores)
print(retrieved_docs[0][:500])


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-small-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


What are some of the primary areas where machine learning can play an important role?
[0.9225807189941406, 0.9089112281799316]
are the primary areas where machine learning can play an important role. - Education Though a teacher’s primary purpose is to teach the subject allocated to him/her to the students. However, he/she cannot be sure whether an entire lot of students are able to understand the concepts or not. With the aid and assistance of machine learning, a teacher can get deep insights and statistics about a student through his/her performance. Machine Learning can help to know how much a student is able to lear



В качестве генератора используется небольшая instruction-tuned модель, которую можно применять для инференса и затем дообучить на выбранных ответах. Prompt формируется явно: в него отдельно передаются контекст, вопрос и инструкция к ответу.


In [11]:
SYSTEM_PROMPT = (
    "You answer questions using only the provided document. "
    "Keep the answer short. If the document is not enough, say so plainly."
)

GENERATOR_MAX_INPUT_TOKENS = 768

class Generator:
    def __init__(
        self,
        model_name: str = CONFIG["generation_model_name"],
        device: str = GEN_DEVICE,
        max_input_tokens: int = GENERATOR_MAX_INPUT_TOKENS,
    ):
        self.device = device
        self.max_input_tokens = max_input_tokens
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.padding_side = "left"

        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            dtype=pick_torch_dtype(device),
            low_cpu_mem_usage=True,
        )
        self.model.to(device)
        for attr in ("temperature", "top_p", "top_k"):
            if hasattr(self.model.generation_config, attr):
                setattr(self.model.generation_config, attr, None)
        self.model.eval()

    def generate(self, inputs: list[str], **generation_kwargs) -> list[str]:
        tokenized = self.tokenizer(
            inputs,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=self.max_input_tokens,
        )
        prompt_lengths = tokenized["attention_mask"].sum(dim=1).tolist()
        if len(set(prompt_lengths)) > 1:
            answers = []
            for text in inputs:
                answers.extend(self.generate([text], **generation_kwargs))
            return answers

        tokenized = {key: value.to(self.device) for key, value in tokenized.items()}
        with torch.inference_mode():
            generated = self.model.generate(
                **tokenized,
                pad_token_id=self.tokenizer.pad_token_id,
                **generation_kwargs,
            )
        generated_only = generated[:, tokenized["input_ids"].shape[1]:]
        answers = self.tokenizer.batch_decode(generated_only, skip_special_tokens=True)
        return [answer.strip() for answer in answers]


def build_rag_prompt(question: str, document: str) -> str:
    return (
        f"Instruction: {SYSTEM_PROMPT}\n\n"
        f"Document:\n{document.strip()}\n\n"
        f"Question: {question.strip()}\n"
        "Answer:"
    )


In [12]:
generator = Generator()

demo_context, _ = vse.get_k_most_similar(train_queries[0], k=1)
demo_prompt = build_rag_prompt(train_queries[0], demo_context[0])
print(demo_prompt)
print("=" * 20)
generator.generate([demo_prompt], max_new_tokens=64, do_sample=False)[0]


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Instruction: You answer questions using only the provided document. Keep the answer short. If the document is not enough, say so plainly.

Document:
are the primary areas where machine learning can play an important role. - Education Though a teacher’s primary purpose is to teach the subject allocated to him/her to the students. However, he/she cannot be sure whether an entire lot of students are able to understand the concepts or not. With the aid and assistance of machine learning, a teacher can get deep insights and statistics about a student through his/her performance. Machine Learning can help to know how much a student is able to learn and understand, the best methods of teaching, students requiring special focus etc. - Digital Marketing

Question: What are some of the primary areas where machine learning can play an important role?
Answer:


"The primary areas where machine learning can play an important role include education and digital marketing. These fields require teachers to assess their students' understanding and provide personalized instruction based on that assessment. Additionally, machine learning algorithms can analyze data from various sources to identify patterns and trends in student behavior, helping educators tailor their teaching strategies effectively."

# Генерация гипотез

Для каждого запроса создаётся несколько вариантов ответа: один greedy-ответ и несколько sample-кандидатов. Это нужно для последующего отбора лучшего ответа через reward-модель.


In [13]:
def create_hypotheses(
    prompt: str,
    generator: Generator,
    temperature: float = 0.8,
    top_p: float = 0.9,
    max_new_tokens: int = 64,
    n_candidates: int = 3,
) -> tuple[str, list[str]]:
    greedy_answer = generator.generate(
        [prompt],
        max_new_tokens=max_new_tokens,
        do_sample=False,
    )[0]
    sampled_answers = generator.generate(
        [prompt] * n_candidates,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
    )
    return greedy_answer, [answer.strip() for answer in sampled_answers]


In [14]:
greedy_answer, sampled_answers = create_hypotheses(
    demo_prompt,
    generator,
    n_candidates=CONFIG["n_candidates"]
)
print("greedy:", greedy_answer)
print()
print("sample:", sampled_answers[0])


greedy: The primary areas where machine learning can play an important role include education and digital marketing. These fields require teachers to assess their students' understanding and provide personalized instruction based on that assessment. Additionally, machine learning algorithms can analyze data from various sources to identify patterns and trends in student behavior, helping educators tailor their teaching strategies effectively.

sample: The primary areas where machine learning can play an important role include education and digital marketing. Specifically, it can assist teachers in understanding students' comprehension levels and providing personalized lesson plans, as well as identifying which students require more focused instruction. Additionally, it allows for analyzing data to identify trends and patterns that can inform educational practices and


Reward model

Reward-модель используется для оценки качества сгенерированных ответов. В этом пайплайне она работает pointwise: каждому кандидату присваивается score, после чего для каждого запроса выбирается лучший вариант.


In [15]:
class RewardModel:
    def __init__(self, model_name: str = CONFIG["reward_model_name"], device: str = REWARD_DEVICE):
        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            dtype=pick_torch_dtype(device),
            low_cpu_mem_usage=True,
        )
        self.model.to(device)
        self.model.eval()

    def _format_conversation(self, prompt: str, answer: str) -> str:
        conversation = [
            {"role": "user", "content": prompt.strip()},
            {"role": "assistant", "content": answer.strip()},
        ]
        formatted = self.tokenizer.apply_chat_template(conversation, tokenize=False)
        if self.tokenizer.bos_token is not None and formatted.startswith(self.tokenizer.bos_token):
            formatted = formatted[len(self.tokenizer.bos_token):]
        return formatted

    def get_score(self, prompt: str, reference_answer: str, candidate_answer: str) -> float:
        _ = reference_answer # используется для API совместимости
        return self.get_scores(prompt, reference_answer, [candidate_answer])[0]

    def get_scores(self, prompt: str, reference_answer: str, candidate_answers: list[str]) -> list[float]:
        _ = reference_answer # используется для API совместимости
        formatted_conversations = [
            self._format_conversation(prompt, candidate_answer)
            for candidate_answer in candidate_answers
        ]
        tokenized = self.tokenizer(
            formatted_conversations,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=2048,
        ).to(self.device)
        with torch.inference_mode():
            logits = self.model(**tokenized).logits.reshape(-1)
        return logits.float().cpu().tolist()


reward_model = RewardModel(model_name=CONFIG["reward_model_name"], device=REWARD_DEVICE)


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/500 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

In [16]:
reward_scores = reward_model.get_scores(
    demo_prompt,
    train_answers[0],
    [greedy_answer] + sampled_answers
)

list(zip(["greedy"] + ["sample"] * len(sampled_answers), reward_scores))

[('greedy', -2.520292043685913),
 ('sample', -1.2768441438674927),
 ('sample', -0.8503905534744263),
 ('sample', -1.9961134195327759),
 ('sample', -5.262183666229248),
 ('sample', -6.048163414001465)]

Сбор кандидатов и Rejection Sampling

На этом этапе объединяются retrieval, генерация и reward scoring: для каждого вопроса извлекается контекст, генерируются кандидаты и считается reward score. Результаты сохраняются в `candidate_df`, который затем используется для отбора лучших ответов и подготовки SFT-датасета.


In [17]:
subset_size = CONFIG["subset_size"] # Можешь менять после тестов
train_df = pick_long_context_examples(wikiq["train"], subset_size)

train_full_docs = train_df["context"].tolist()
train_queries = train_df["question"].tolist()
train_answers = train_df["answer"].tolist()

chunker = Chunker(chunk_words=100, overlap=20)
train_chunked_docs = chunker.get_chunked_list(train_full_docs)

encoder = Encoder()
vse = VectorSearchEngine(train_chunked_docs, encoder)

generator = Generator()
reward_model = RewardModel()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-small-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

In [18]:
CANDIDATES_PATH = ARTIFACTS_DIR / "candidate_rows.jsonl"
BEST_ROWS_PATH = ARTIFACTS_DIR / "best_rows.jsonl"
TRAIN_OUTPUT_DIR = ARTIFACTS_DIR / "training_output"
FINAL_MODEL_DIR = ARTIFACTS_DIR / "trained_generator"

# Можешь менять после тестов
retrieval_top_k = 1
candidate_save_every = 1
hypothesis_temperature = 0.8
hypothesis_top_p = 0.9
hypothesis_max_new_tokens = 64
hypothesis_n_candidates = CONFIG["n_candidates"]
force_rebuild_candidates = False


def _json_default(value):
    if hasattr(value, "item"):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f"Object of type {type(value)!r} is not JSON serializable")


def write_jsonl(path: Path, rows: list[dict]) -> None:
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False, default=_json_default) + "\n")


def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def build_candidate_frame(
    samples_df: pd.DataFrame,
    search_engine: VectorSearchEngine,
    generator: Generator,
    reward_model: RewardModel,
    artifact_path: Path,
    resume: bool = True,
    save_every: int = candidate_save_every,
) -> pd.DataFrame:
    rows = read_jsonl(artifact_path) if resume and artifact_path.exists() else []
    completed_ids = {row["example_id"] for row in rows if row["generation_type"] == "greedy"}

    for sample in tqdm(samples_df.to_dict("records"), total=len(samples_df)):
        example_id = int(sample["example_id"])
        if example_id in completed_ids:
            continue

        retrieved_docs, _ = search_engine.get_k_most_similar(
            sample["question"],
            k=retrieval_top_k,
        )
        retrieved_context = retrieved_docs[0]
        prompt = build_rag_prompt(sample["question"], retrieved_context)

        greedy_answer, sampled_answers = create_hypotheses(
            prompt,
            generator,
            temperature=hypothesis_temperature,
            top_p=hypothesis_top_p,
            max_new_tokens=hypothesis_max_new_tokens,
            n_candidates=hypothesis_n_candidates,
        )
        all_answers = [greedy_answer] + sampled_answers
        reward_scores = reward_model.get_scores(prompt, sample["answer"], all_answers)

        current_rows = []
        for generation_type, answer_text, reward_score in zip(
            ["greedy"] + ["sample"] * len(sampled_answers),
            all_answers,
            reward_scores,
        ):
            current_rows.append(
                {
                    "example_id": example_id,
                    "question": sample["question"],
                    "reference_answer": sample["answer"],
                    "retrieved_context": retrieved_context,
                    "generation": answer_text,
                    "generation_type": generation_type,
                    "reward_score": float(reward_score),
                }
            )

        rows.extend(current_rows)
        completed_ids.add(example_id)

        if len(completed_ids) % save_every == 0:
            write_jsonl(artifact_path, rows)

    write_jsonl(artifact_path, rows)
    frame = pd.DataFrame(rows)
    frame = frame.sort_values(["example_id", "generation_type", "reward_score"], ascending=[True, True, False])
    return frame.reset_index(drop=True)


In [19]:
if force_rebuild_candidates and CANDIDATES_PATH.exists():
    CANDIDATES_PATH.unlink()

candidate_df = build_candidate_frame(
    train_df,
    vse,
    generator,
    reward_model,
    CANDIDATES_PATH,
    resume=not force_rebuild_candidates,
    save_every=candidate_save_every,
)

print(f"rows: {len(candidate_df)}")
print(
    f"greedy mean reward: "
    f"{candidate_df.loc[candidate_df['generation_type'] == 'greedy', 'reward_score'].mean():.4f}"
)
candidate_df.head(3)


  0%|          | 0/6 [00:00<?, ?it/s]

rows: 36
greedy mean reward: -1.7989


,example_id,question,reference_answer,retrieved_context,generation,generation_type,reward_score
0,0,What are some of the primary areas where machi...,The primary areas where machine learning can p...,are the primary areas where machine learning c...,The primary areas where machine learning can p...,greedy,-2.520292
1,0,What are some of the primary areas where machi...,The primary areas where machine learning can p...,are the primary areas where machine learning c...,The primary areas where machine learning can p...,sample,0.342136
2,0,What are some of the primary areas where machi...,The primary areas where machine learning can p...,are the primary areas where machine learning c...,The primary areas where machine learning can p...,sample,-4.037629


Rejection sampling используется как простой способ получить более чистый обучающий набор: из нескольких сгенерированных ответов выбирается кандидат с максимальным reward score. После отбора можно сравнить средний reward выбранных ответов с greedy baseline.


In [20]:
best_score_df = (
    candidate_df
    .sort_values(["example_id", "reward_score"], ascending=[True, False])
    .groupby("example_id", as_index=False)
    .first()
)

write_jsonl(BEST_ROWS_PATH, best_score_df.to_dict(orient="records"))

greedy_mean = candidate_df.loc[candidate_df["generation_type"] == "greedy", "reward_score"].mean()
rs_mean = best_score_df["reward_score"].mean()

print(f"greedy mean reward: {greedy_mean:.4f}")
print(f"rejection sampling mean reward: {rs_mean:.4f}")
best_score_df[["example_id", "question", "generation", "reward_score"]].head(3)


greedy mean reward: -1.7989
rejection sampling mean reward: -0.1252


,example_id,question,generation,reward_score
0,0,What are some of the primary areas where machi...,The primary areas where machine learning can p...,0.342136
1,1,What are some methods used to align teeth?,Dental braces are often used for tooth alignme...,0.077616
2,2,What is the rating of the Cheap School Movie R...,The movie review has a rating of 4 out of 5 ba...,4.044599


Короткий SFT-цикл

Выбранные через rejection sampling ответы используются как небольшой SFT-датасет для генератора. Цель этого этапа — замкнуть полный цикл `retrieve → generate → score → select → fine-tune` и проверить, что пайплайн технически работает end-to-end.


In [21]:
def format_training_example(row: pd.Series) -> str:
    prompt = build_rag_prompt(row["question"], row["retrieved_context"])
    return f"{prompt} {row['generation'].strip()}"


best_score_df = best_score_df.copy()
best_score_df["train_text"] = best_score_df.apply(format_training_example, axis=1)

train_dataset = Dataset.from_pandas(
    best_score_df[["train_text"]].rename(columns={"train_text": "text"}),
    preserve_index=False,
)
print(train_dataset[0]["text"][:600])


Instruction: You answer questions using only the provided document. Keep the answer short. If the document is not enough, say so plainly.

Document:
are the primary areas where machine learning can play an important role. - Education Though a teacher’s primary purpose is to teach the subject allocated to him/her to the students. However, he/she cannot be sure whether an entire lot of students are able to understand the concepts or not. With the aid and assistance of machine learning, a teacher can get deep insights and statistics about a student through his/her performance. Machine Learning ca


Для короткого SFT используется `trl.SFTTrainer` с консервативными параметрами обучения. Модель для обучения загружается отдельно от инференс-модели, а checkpoint-ы сохраняются, чтобы запуск можно было продолжить после сбоя среды.


In [ ]:
train_max_length = 384
per_device_train_batch_size = 1
gradient_accumulation_steps = 2 # Если GPU
train_max_steps = 6
learning_rate = 3e-5
save_steps = 2
force_rebuild_training = False

if force_rebuild_training:
    if TRAIN_OUTPUT_DIR.exists():
        shutil.rmtree(TRAIN_OUTPUT_DIR)
    if FINAL_MODEL_DIR.exists():
        shutil.rmtree(FINAL_MODEL_DIR)

TRAIN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
last_checkpoint = get_last_checkpoint(str(TRAIN_OUTPUT_DIR)) if not force_rebuild_training else None

if hasattr(generator, "model"):
    del generator.model
cleanup_memory()

train_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["generation_model_name"],
    dtype=torch.float32,
    low_cpu_mem_usage=True,
)
train_model.to(GEN_DEVICE)
train_model.train()
train_model.config.use_cache = False

sft_args = SFTConfig(
    output_dir=str(TRAIN_OUTPUT_DIR),
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    max_steps=train_max_steps,
    learning_rate=learning_rate,
    save_strategy="steps",
    save_steps=save_steps,
    save_total_limit=2,
    logging_steps=1,
    report_to="none",
    seed=CONFIG["seed"],
    dataloader_pin_memory=use_pin_memory(GEN_DEVICE),
    gradient_checkpointing=use_gradient_checkpointing(GEN_DEVICE),
    fp16=use_fp16_training(GEN_DEVICE),
    bf16=use_bf16_training(GEN_DEVICE),
    dataset_text_field="text",
    max_length=train_max_length,
    packing=False,
)

trainer = SFTTrainer(
    model=train_model,
    args=sft_args,
    train_dataset=train_dataset,
    processing_class=generator.tokenizer,
)

train_result = trainer.train(resume_from_checkpoint=last_checkpoint if last_checkpoint else None)
trainer.save_model(str(FINAL_MODEL_DIR))
generator.tokenizer.save_pretrained(str(FINAL_MODEL_DIR))

# Trainer тоже держит ссылку на train-time модель, поэтому освобождаем оба объекта
# до post-train evaluation.
del trainer
del train_model
cleanup_memory()

print(train_result.metrics)
print(f"saved model: {FINAL_MODEL_DIR}")


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
1,2.777552
2,3.349649
3,2.810936
4,1.210732
5,1.804635
6,1.675870


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': 661.579, 'train_samples_per_second': 0.018, 'train_steps_per_second': 0.009, 'total_flos': 5660516539392.0, 'train_loss': 2.2715624570846558}
saved model: rag_artifacts/trained_generator




После короткого SFT выполняется базовая проверка: генератор снова отвечает на тот же набор запросов, а средний reward сравнивается с greedy baseline до обучения. Это не полноценная offline evaluation, а быстрый sanity check работоспособности пайплайна.


In [23]:
for variable_name in ("reward_model", "generator", "encoder", "vse", "train_dataset"):
    if variable_name in globals():
        del globals()[variable_name]
cleanup_memory()

eval_max_new_tokens = 64

trained_generator = Generator(
    model_name=str(FINAL_MODEL_DIR),
    device=GEN_DEVICE,
    max_input_tokens=GENERATOR_MAX_INPUT_TOKENS,
)
reward_model = RewardModel(model_name=CONFIG["reward_model_name"], device=REWARD_DEVICE)

baseline_greedy_df = candidate_df[candidate_df["generation_type"] == "greedy"].sort_values("example_id")

eval_rows = []
for row in tqdm(baseline_greedy_df.to_dict("records"), total=len(baseline_greedy_df)):
    prompt = build_rag_prompt(row["question"], row["retrieved_context"])
    answer = trained_generator.generate(
        [prompt],
        max_new_tokens=eval_max_new_tokens,
        do_sample=False,
    )[0]
    eval_rows.append(
        {
            "example_id": row["example_id"],
            "question": row["question"],
            "generation": answer,
            "reward_score": reward_model.get_score(prompt, row["reference_answer"], answer),
        }
    )

eval_df = pd.DataFrame(eval_rows)
print(f"baseline greedy mean reward: {baseline_greedy_df['reward_score'].mean():.4f}")
print(f"trained greedy mean reward: {eval_df['reward_score'].mean():.4f}")
eval_df.head(3)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

  0%|          | 0/6 [00:00<?, ?it/s]

baseline greedy mean reward: -1.7989
trained greedy mean reward: -0.8368


,example_id,question,generation,reward_score
0,0,What are some of the primary areas where machi...,The primary areas where machine learning can p...,0.342136
1,1,What are some methods used to align teeth?,Dental braces are commonly used for tooth alig...,-5.171382
2,2,What is the rating of the Cheap School Movie R...,The movie review has a rating of 4 out of 5 ba...,4.044594


В конце показан простой toy-пример query expansion: к словам запроса добавляются синонимы, после чего сравнивается выдача retrieval до и после расширения. В реальных системах такой подход требует отдельной проверки, потому что он может как улучшить, так и ухудшить качество поиска.


In [24]:
import ssl

import certifi
import nltk
from nltk.corpus import wordnet

WORDNET_READY = False
LOCAL_FALLBACK_SYNONYMS = {
    "movie": "film",
    "prizes": "awards",
}


def ensure_wordnet_ready() -> bool:
    global WORDNET_READY
    if WORDNET_READY:
        return True
    try:
        wordnet.ensure_loaded()
        WORDNET_READY = True
        return True
    except LookupError:
        try:
            ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())
            download_ok = nltk.download("wordnet", quiet=True)
            nltk.download("omw-1.4", quiet=True)
            if not download_ok:
                raise LookupError("nltk.download('wordnet') returned False")
            wordnet.ensure_loaded()
            WORDNET_READY = True
            print("Downloaded WordNet locally for query expansion.")
        except Exception as exc:
            print(
                f"WordNet is unavailable locally and could not be downloaded ({exc}). "
                "The toy example will use a tiny local synonym fallback."
            )
    return WORDNET_READY


ensure_wordnet_ready()

Downloaded WordNet locally for query expansion.


True

In [25]:
def pick_synonym(word: str) -> str | None:
    normalized = word.lower().strip(".,!?;:")
    if not normalized:
        return None
    if WORDNET_READY:
        for synset in wordnet.synsets(normalized):
            for lemma in synset.lemmas():
                candidate = lemma.name().replace("_", " ")
                if candidate.lower() != normalized:
                    return candidate
    return LOCAL_FALLBACK_SYNONYMS.get(normalized)


def expand_query(query: str) -> str:
    expanded_words = []
    for word in query.split():
        expanded_words.append(word)
        synonym = pick_synonym(word)
        if synonym:
            expanded_words.append(synonym)
    return " ".join(expanded_words)


def inspect_retrieval(query_text: str, search_engine: VectorSearchEngine, k: int = 2) -> tuple[list[str], list[float]]:
    docs, scores = search_engine.get_k_most_similar(query_text, k=k)
    print(f"query: {query_text}")
    for rank, (doc, score) in enumerate(zip(docs, scores), start=1):
        print(f"top-{rank} | score={score:.4f} | {doc}")
    return docs, scores


documents = [
    "The film won multiple festival prizes and received strong critic reviews.",
    "A movie theater sells snacks and tickets near the city center.",
    "A prize catalog lists household items and gift cards.",
]

toy_encoder = Encoder(model_name=CONFIG["retrieval_model_name"], device=EMBED_DEVICE)
toy_vse = VectorSearchEngine(documents, toy_encoder)

query = "movie prizes"
expanded_query = expand_query(query)

print("original:", query)
print("expanded:", expanded_query)
print()

original_docs, _ = inspect_retrieval(query, toy_vse, k=2)
print()
expanded_docs, _ = inspect_retrieval(expanded_query, toy_vse, k=2)
print()

top1_changed = original_docs[0] != expanded_docs[0]
print("top-1 changed:", top1_changed)
if top1_changed:
    print("interpretation: expansion added wording that better matched the relevant document.")
else:
    print("interpretation: the original query was already strong enough or the added synonym did not help here.")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/e5-small-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


original: movie prizes
expanded: movie film prizes prize

query: movie prizes
top-1 | score=0.8001 | A movie theater sells snacks and tickets near the city center.
top-2 | score=0.7998 | A prize catalog lists household items and gift cards.

query: movie film prizes prize
top-1 | score=0.8039 | The film won multiple festival prizes and received strong critic reviews.
top-2 | score=0.7597 | A prize catalog lists household items and gift cards.

top-1 changed: True
interpretation: expansion added wording that better matched the relevant document.
